# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily_march': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}
print("Connected.")

Connected.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= '2026-03-20' THEN gsc_impressions ELSE 0 END) AS imp_early,
            SUM(CASE WHEN report_date > '2026-03-20' THEN gsc_impressions ELSE 0 END) AS imp_late,
            AVG(CASE WHEN report_date <= '2026-03-20' THEN gsc_avg_position END) AS pos_early,
            AVG(CASE WHEN report_date <= '2026-03-20' THEN gsc_clicks END) AS clicks_early_avg
        FROM {TABLES['fact_daily_march']}
        GROUP BY 1,2
        HAVING imp_early >= 50
    )
    SELECT * FROM agg
""").df()

# Categorical/missing handling
features['pos_early'] = features['pos_early'].fillna(999)  # 999 = "not ranked" sentinel
features['clicks_early_avg'] = features['clicks_early_avg'].fillna(0)

# Build the label from the LATE window only, never mixed into features
features['rate_early'] = features['imp_early'] / 20
features['rate_late'] = features['imp_late'] / 11
features['is_declining'] = (features['rate_late'] < 0.8 * features['rate_early']).astype(int)

print(f"{len(features):,} content items with enough early-window volume")
print("Decline rate:", features['is_declining'].mean().round(3))
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

102,537 content items with enough early-window volume
Decline rate: 0.324


,client_hash_id,content_hash_id,imp_early,imp_late,pos_early,clicks_early_avg,rate_early,rate_late,is_declining
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,5131.0,1392.0,6.880718,0.30,256.55,126.545455,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,301.0,152.0,3.481726,0.00,15.05,13.818182,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,4456.0,1174.0,6.603060,0.15,222.80,106.727273,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,3289.0,1655.0,7.242370,0.60,164.45,150.454545,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,340.0,89.0,3.679318,0.05,17.00,8.090909,1


I split March into two windows: days 1-20 (feature window) and days
21-31 (label window). Features are aggregated per content item from
the feature window only - nothing from the label window touches the
features, to keep the split honest.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(features[['imp_early','pos_early','clicks_early_avg']].describe())

           imp_early      pos_early  clicks_early_avg
count  102537.000000  102537.000000     102537.000000
mean     1657.546573      14.526900          0.254142
std      4157.127333      14.882267          1.071076
min        50.000000       0.000000          0.000000
25%       155.000000       4.806065          0.000000
50%       455.000000       8.582436          0.050000
75%      1518.000000      19.084762          0.200000
max    244485.000000     112.213421        160.200000


- **imp_early** (total impressions, days 1-20): meaning = search
  visibility in the feature window. Missing: none (SUM defaults to 0).
  Available-when: fully known by day 20, before the label window starts.
- **pos_early** (avg search position, days 1-20): meaning = how well
  the page ranks on average. Missing: filled with 999 (sentinel for
  "not ranked/no data"), since a missing position isn't the same as
  position 0. Available-when: known by day 20.
- **clicks_early_avg** (avg daily clicks, days 1-20): meaning =
  click volume trend. Missing: filled with 0 (no clicks recorded =
  reasonable default). Available-when: known by day 20.
- **rate_early** (derived: imp_early / 20): daily impression rate in
  the feature window - used to fairly compare against the label window's
  different day count. Available-when: same as imp_early.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feat_cols = ['imp_early', 'pos_early', 'clicks_early_avg', 'rate_early']
X, y = features[feat_cols], features['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
print(f"Honest AUC (no leakage): {honest_auc:.3f}")

Honest AUC (no leakage): 0.563


Honest baseline first, then I deliberately add rate_late (built from
the LABEL window itself) as a feature - the leak - and watch the score
jump toward a suspiciously perfect number. Then I remove it.

In [7]:
feat_cols_leaky = feat_cols + ['rate_late']  # rate_late IS the label's source - leaking on purpose
X_leak = features[feat_cols_leaky]
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)

leaky_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xl_tr, yl_tr)
leaky_auc = roc_auc_score(yl_te, leaky_model.predict_proba(Xl_te)[:,1])
print(f"'Leaky' AUC (rate_late included): {leaky_auc:.3f}  <- suspiciously high")
print(f"Honest AUC was: {honest_auc:.3f}")

'Leaky' AUC (rate_late included): 1.000  <- suspiciously high
Honest AUC was: 0.563


As expected, adding `rate_late` - a value computed directly from the
same window the label is derived from - pushes AUC toward a
near-perfect score. This is fake: the model isn't learning a real
pattern, it's just reading the answer. `rate_late` is removed and only
the honest feature set is used going forward.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Final honest feature set used:", feat_cols)
print("Excluded as leakage:", ['rate_late', 'imp_late'])

Final honest feature set used: ['imp_early', 'pos_early', 'clicks_early_avg', 'rate_early']
Excluded as leakage: ['rate_late', 'imp_late']


- **rate_late / imp_late:** directly derived from the label window -
  this is the label in disguise, proven by the leakage test above.
- **health_score, priority_score, action_type:** FlyRank product
  decision flags, not present in this table and not to be reconstructed.
- **ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude,
  ai_meta, ai_other:** AI-referral columns - sparse per the lane guide,
  safe for EDA only, not as classifier features.
- **client_hash_id, content_hash_id, report_date:** join/identity keys,
  not signal - used for grouping only, never as model input.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.